# VeriGym - Comparing exploration policies

In this workflow, we zoom in on **exploration policies**, which are a critical component of model learning. We show off and test the 4 exploration policies that are natively supported by `VeriGym`. Furthermore, we show how you can define your own exploration policy and use it in the same way.

## 1. Setup

We start by setting up our environment and abstraction: for more details, see the specific workflows on this. In this file, we use the Gymnasiums `MountainCar` environment which, as we'll see later, requires smart exploration to learn.

In [1]:
import stormpy
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import logging

import verigym
from verigym.abstraction.gym_utils.transform_observation import ReplaceInfObservation
from verigym.abstraction.abstractionmapper import linspace_mapper
from verigym.frameworks.stormpy.stormpy_utils import build_stormpy_mdp
from verigym.frameworks.stormpy.stormpypolicy import StormpyPolicy
from verigym.policy.randomized import RandomizedPolicy
from verigym.policy.policy import PolicyClass
from verigym.policy.qvalue import (
    QValuePolicy,
    ActiveLearningPolicy,
    EntropyLearningPolicy,
)

from workflow_utils import get_explored_state_action_pairs, get_mean_reward_from_trajectories
logging.getLogger().setLevel(logging.ERROR)


In [2]:
# 1. Load MountainCar environment
gym_env = gym.make("MountainCar-v0")
print("Environment type: ", type(gym_env))

# Create a VeriGymEnv from gym env
generative_model = verigym.GenerativeEnv.from_gymnasium(gym_env)

print("Environment type: ", type(generative_model))
print("Observation space shape: ", generative_model.observation_space.shape)
print("Type of observation space: ", type(generative_model.observation_space))

# Define abstraction
abstraction_mapper = linspace_mapper(
    env=generative_model,
    n_bins_states=20,
    n_bins_actions=3,
)

print(f"The abstract observation space has {abstraction_mapper.abstract_n_states} states!")
print(f"The abstract action space has {abstraction_mapper.abstract_n_actions} actions!")

Environment type:  <class 'gymnasium.wrappers.common.TimeLimit'>
Environment type:  <class 'verigym.environments.generativeenv.GenerativeEnv_from_gym'>
Observation space shape:  (2,)
Type of observation space:  <class 'gymnasium.spaces.box.Box'>
The abstract observation space has 400 states!
The abstract action space has 3 actions!


## 2. Exploration policies

`VeriGym` makes use of **exploration policies** to collect samples from the environment for model learning. We natively support a number of methods to do this:

1. `RandomizedPolicy`: a policy that uniformly samples actions.
2. `QValuePolicy`: uses an optimistic version of abstraction learned *so far* to estimate Q-values. Actions are picked epsilon-greedily based on these values.
3. `ActiveLearningPolicy`: based Araya-López et al. (2012). Computes Q-values similarly to `QValuePolicy`, but uses an *exploration reward* inversely proportional to the visitation count instead of the true reward function.
4. `EntropyLearningPolicy`: based on Hazan et al. (2019). Iteratively computes a maximum-entropy policy over the state distribution.

We can define them all as follows:

In [3]:
epsilon = 0.1 # exploration threshold for the epsilon-greedy policies

# Constructors, such that every abstraction starts from a fresh policy
exploration_policies = {
    "Randomized": RandomizedPolicy(generative_model),
    "QValue": QValuePolicy(generative_model, abstraction_mapper, epsilon=epsilon),
    "ActiveLearning": ActiveLearningPolicy(generative_model, abstraction_mapper, epsilon=epsilon),
    "EntropyLearning": EntropyLearningPolicy(generative_model, abstraction_mapper),
}

During model learning, these policies should periodically be updated based on the current best estimate of the abstracted model. Such updates are implemented in the function `update_for_abstraction_refinment(...)`, which takes as input the data gathered thus far. For example, when calling this function on `QValuePolicy`, it updates it's estimate of the model, and then updates it's Q-values via a number of value iteration steps on this model.

When calling `create_abstraction`, the frequency of such policy updates can be determined by the parameter `n_iterations`. Each iteration consists of `num_steps` interactions with the environment, after which the policy is updated. More frequent updates yield better exploration behaviour, but requires more (computationally expensive) policy updates. For our example, we use the following parameters:

In [4]:
n_iterations = 50
num_steps = int(2e4)

In total, we thus have a budget of `n_iterations` $*$ `num_steps` $= 500\_000$ interations with the environment.

## 3. Model Learning
Next, let's use the different policies to learn model abstractions via the `create_abstraction` function:

In [5]:
# TODO: record timings, change random to 1 iteration

abstracted_models = {}
for name, make_policy in exploration_policies.items():
    print(f"--- {name} ---")
    abstracted_models[name] = verigym.create_abstraction(
        original_env=generative_model,
        abstraction_mapper=abstraction_mapper,
        exploration_policy=make_policy,
        num_steps=num_steps,
        n_iterations=n_iterations,
        multithreading=False
    )

--- Randomized ---
Simulation time: 0.4595s
Trajectories in dataset: 100
Learning Abstraction: 0.6077s
Simulation time: 0.5102s
Trajectories in dataset: 100
Learning Abstraction: 0.6472s
Simulation time: 0.5105s
Trajectories in dataset: 100
Learning Abstraction: 0.5950s
Simulation time: 0.4805s
Trajectories in dataset: 100
Learning Abstraction: 0.6017s
Simulation time: 0.4492s
Trajectories in dataset: 100
Learning Abstraction: 0.5475s
Simulation time: 0.4268s
Trajectories in dataset: 100
Learning Abstraction: 0.5503s
Simulation time: 0.4828s
Trajectories in dataset: 100
Learning Abstraction: 0.6882s
Simulation time: 0.4625s
Trajectories in dataset: 100
Learning Abstraction: 0.5622s
Simulation time: 0.4257s
Trajectories in dataset: 100
Learning Abstraction: 0.5327s
Simulation time: 0.4122s
Trajectories in dataset: 100
Learning Abstraction: 0.5614s
Simulation time: 0.4848s
Trajectories in dataset: 100
Learning Abstraction: 0.5897s
Simulation time: 0.4373s
Trajectories in dataset: 100
Lea

/home/merlijn/GitRepos/VeriGym/src/verigym/abstraction/learn_abstraction.py:384: RuntimeWarning: invalid value encountered in divide
  state_distr /= state_distr.sum()


Simulation time: 2.2053s
Trajectories in dataset: 100
Learning Abstraction: 0.7911s
Simulation time: 2.1225s
Trajectories in dataset: 100
Learning Abstraction: 0.6095s
Simulation time: 2.7833s
Trajectories in dataset: 100
Learning Abstraction: 0.6406s
Simulation time: 2.0231s
Trajectories in dataset: 100
Learning Abstraction: 0.7755s
Simulation time: 1.9262s
Trajectories in dataset: 100
Learning Abstraction: 0.5901s
Simulation time: 1.9794s
Trajectories in dataset: 100
Learning Abstraction: 0.6357s
Simulation time: 2.5675s
Trajectories in dataset: 100
Learning Abstraction: 0.6709s
Simulation time: 2.1526s
Trajectories in dataset: 100
Learning Abstraction: 0.5973s
Simulation time: 1.8405s
Trajectories in dataset: 100
Learning Abstraction: 0.5407s
Simulation time: 1.7614s
Trajectories in dataset: 100
Learning Abstraction: 0.6239s
Simulation time: 2.0049s
Trajectories in dataset: 100
Learning Abstraction: 0.6663s
Simulation time: 3.1889s
Trajectories in dataset: 100
Learning Abstraction: 

/home/merlijn/GitRepos/VeriGym/src/verigym/abstraction/learn_abstraction.py:384: RuntimeWarning: invalid value encountered in divide
  state_distr /= state_distr.sum()


Simulation time: 2.0263s
Trajectories in dataset: 100
Learning Abstraction: 0.8308s
Simulation time: 2.0551s
Trajectories in dataset: 100
Learning Abstraction: 0.5703s
Simulation time: 1.8393s
Trajectories in dataset: 100
Learning Abstraction: 0.5459s
Simulation time: 1.7418s
Trajectories in dataset: 100
Learning Abstraction: 0.5001s
Simulation time: 1.6908s
Trajectories in dataset: 100
Learning Abstraction: 0.5141s
Simulation time: 1.9779s
Trajectories in dataset: 100
Learning Abstraction: 0.6117s
Simulation time: 2.0139s
Trajectories in dataset: 100
Learning Abstraction: 0.5408s
Simulation time: 1.8164s
Trajectories in dataset: 100
Learning Abstraction: 0.5398s
Simulation time: 1.7959s
Trajectories in dataset: 100
Learning Abstraction: 0.5555s
Simulation time: 1.8974s
Trajectories in dataset: 100
Learning Abstraction: 0.8150s
Simulation time: 1.9480s
Trajectories in dataset: 100
Learning Abstraction: 0.5404s
Simulation time: 1.8167s
Trajectories in dataset: 100
Learning Abstraction: 

/home/merlijn/GitRepos/VeriGym/src/verigym/policy/qvalue.py:226: RuntimeWarning: divide by zero encountered in log
  R_learning[sidx][aidx] = min(Rmax, -(np.log(d_pi[sidx]) + 1))


Simulation time: 1.8219s
Trajectories in dataset: 100
Learning Abstraction: 0.7812s
Simulation time: 1.4809s
Trajectories in dataset: 100
Learning Abstraction: 0.5814s
Simulation time: 1.7010s
Trajectories in dataset: 100
Learning Abstraction: 0.8001s
Simulation time: 1.7526s
Trajectories in dataset: 100
Learning Abstraction: 0.6088s
Simulation time: 1.5131s
Trajectories in dataset: 100
Learning Abstraction: 0.7577s
Simulation time: 1.3753s
Trajectories in dataset: 101
Learning Abstraction: 0.5653s
Simulation time: 1.3955s
Trajectories in dataset: 103
Learning Abstraction: 0.5481s
Simulation time: 1.3994s
Trajectories in dataset: 105
Learning Abstraction: 0.5774s
Simulation time: 1.5791s
Trajectories in dataset: 112
Learning Abstraction: 0.6135s
Simulation time: 1.4164s
Trajectories in dataset: 116
Learning Abstraction: 0.5768s
Simulation time: 1.7428s
Trajectories in dataset: 112
Learning Abstraction: 0.5610s
Simulation time: 1.2994s
Trajectories in dataset: 111
Learning Abstraction: 

To determine if the policies are any good, let us first compute the **coverage** of each policy: how many states and state-action pairs were explored during model learning?

In [6]:
coverage = {
    name: get_explored_state_action_pairs(model)
    for name, model in abstracted_models.items()
}

n_state_action_pairs = abstraction_mapper.abstract_n_states * abstraction_mapper.abstract_n_actions
print(f"{'Policy':<16} {'States':>8} {'State-action pairs':>20}")
for name, (n_states, n_pairs) in coverage.items():
    print(f"{name:<16} {n_states:>8} {n_pairs:>20}")
print(f"{'Total':<16} {abstraction_mapper.abstract_n_states:>8} {n_state_action_pairs:>20}")

Policy             States   State-action pairs
Randomized             92                  265
QValue                162                  463
ActiveLearning        150                  439
EntropyLearning       294                  882
Total                 400                 1200


We see that the naive `RandomizedPolicy` explores only a small part of the environment, even after a large number of interactions. In contrast, our other exploration policies discover a much larger portion of the state-action space, and thus provide much more useful abstract models. Next, let's see how policies learned on the different models compare. We start by using `stormpy` to compute a policy:

In [9]:
gamma = 0.95  # discount factor
prop = stormpy.parse_properties(f"Rmax=?[Cdiscount={gamma}]")[
    0
]  # checking for cumulative discounted reward

stormpy_mdps = {}
schedulers = {}
for name, abstracted_model in abstracted_models.items():
    stormpy_mdps[name] = build_stormpy_mdp(abstracted_model)
    result = stormpy.check_model_sparse(
        stormpy_mdps[name], prop, extract_scheduler=True
    )
    schedulers[name] = result.scheduler

We then evaluate these policies on both the original and abstracted environment, which yields the following:

In [10]:
n_simulation_steps = int(1e4)

mean_rewards_original = {}
mean_rewards_abstracted = {}
for name, abstracted_model in abstracted_models.items():
    # policy mapped to the original model
    verigym_policy = StormpyPolicy(
        schedulers[name], abstracted_model.abstraction_map, stormpy_mdps[name]
    )
    # policy on the abstract model
    identity_mapper = verigym.AbstractionMapper.initialize_identity_mapper(
        abstracted_model.observation_space, abstracted_model.action_space
    )
    verigym_policy_on_abstracted = StormpyPolicy(
        schedulers[name], identity_mapper, stormpy_mdps[name]
    )

    # simulation in the original (continuous) model
    trajectories_original = generative_model.simulate(
        policy=verigym_policy,
        n_steps=n_simulation_steps,
    )
    mean_rewards_original[name] = get_mean_reward_from_trajectories(
        trajectories_original
    )

    # simulation in the abstracted model
    trajectories_abstracted = abstracted_model.simulate(
        policy=verigym_policy_on_abstracted,
        n_steps=n_simulation_steps,
    )
    mean_rewards_abstracted[name] = get_mean_reward_from_trajectories(
        trajectories_abstracted
    )

Simulating:   0%|          | 0/10000 [00:00<?, ?it/s]

Simulating:   0%|          | 0/10000 [00:00<?, ?it/s]

Simulating:   0%|          | 0/10000 [00:00<?, ?it/s]

Simulating:   0%|          | 0/10000 [00:00<?, ?it/s]

Simulating:   0%|          | 0/10000 [00:00<?, ?it/s]

Simulating:   0%|          | 0/10000 [00:00<?, ?it/s]

Simulating:   0%|          | 0/10000 [00:00<?, ?it/s]

Simulating:   0%|          | 0/10000 [00:00<?, ?it/s]

In [11]:
print("Comparing the mean cumulative reward of the abstract policies:")
print(f"{'Policy':<16} {'Original':>12} {'Abstract':>12}")
for name in abstracted_models:
    print(
        f"{name:<16} {mean_rewards_original[name]:>12.2f} {mean_rewards_abstracted[name]:>12.2f}"
    )

Comparing the mean cumulative reward of the abstract policies:
Policy               Original     Abstract
Randomized            -200.00     -1111.11
QValue                -200.00     -5000.00
ActiveLearning        -200.00    -10000.00
EntropyLearning       -142.86      -140.85


We see that ...

## 4. Defining Exploration Policies

While the natively supported exploration policies are a good starting point, some environments might need more specialized or sophisticated exploration policies to be able to sufficiently sample the entire environments. `VeriGym` supports users to define and use such policies themselves.

As a simple example, we may consider the following heuristic: at each iteration, flip between always trying to increase or decrease the current speed, i.e., flip between accelerating in the same and the oposite direction of the current speed.

We can implement such an policy as a `PolicyClass`. As can be read in more detail in the docs, implementing a specific (exploration) policy class requires implementing the functions `_action_from_policy`, which maps abstracted observations to actions, and `update_for_abstraction_refinement`, which defines how the policy is updated at the end of each iteration. For our example, this implementation looks as follows:

In [ ]:
class CustomRandomPolicy(PolicyClass):
    """
    Custom policy class for MountainCar env. Flips between 
    """

    def _action_from_policy(self, obs):
        velocity = obs[1]
        if self.policy == "speed_up":
            return 0 if velocity < 0 else 2
        else:
            return 0 if velocity > 0 else 2
        return action

    def update_for_abstraction_refinement(self, dataset, T_counts, P_tot_counts, R_dicts_counts, state_distr_counts):
        self.policy = "speed_up" if self.policy == "slow_down" else "slow_down"

With less then 15 lines of codes, we have now defined a custom policy! This policy can immidiately be used in exactly the same way as the native policies we used before, e.g. as an exploration policy in `create_abstraction`:

In [ ]:
custom_policy = CustomRandomPolicy(policy="speed_up", abstraction_mapper=abstraction_mapper)

abstracted_models["Custom"] = verigym.create_abstraction(
        original_env=generative_model,
        abstraction_mapper=abstraction_mapper,
        exploration_policy=custom_policy,
        num_steps=num_steps,
        n_iterations=n_iterations,
        multithreading=False
    )

Simulation time: 0.3489s
Trajectories in dataset: 100
Learning Abstraction: 0.2395s
Simulation time: 0.3741s
Trajectories in dataset: 100
Learning Abstraction: 0.2482s
Simulation time: 0.3596s
Trajectories in dataset: 100
Learning Abstraction: 0.2531s
Simulation time: 0.3680s
Trajectories in dataset: 100
Learning Abstraction: 0.3553s
Simulation time: 0.7368s
Trajectories in dataset: 100
Learning Abstraction: 0.6035s
Simulation time: 0.9201s
Trajectories in dataset: 100
Learning Abstraction: 0.6378s
Simulation time: 0.8550s
Trajectories in dataset: 100
Learning Abstraction: 0.6109s
Simulation time: 0.8592s
Trajectories in dataset: 100
Learning Abstraction: 0.5908s
Simulation time: 0.8649s
Trajectories in dataset: 100
Learning Abstraction: 0.6486s
Simulation time: 0.9004s
Trajectories in dataset: 100
Learning Abstraction: 0.6061s
Simulation time: 0.9700s
Trajectories in dataset: 100
Learning Abstraction: 0.7281s
Simulation time: 0.9166s
Trajectories in dataset: 100
Learning Abstraction: 

Let's see how our custom policy compares to the native ones from before:

In [14]:
coverage["Custom"] = get_explored_state_action_pairs(abstracted_models["Custom"])

print(f"{'Policy':<16} {'States':>8} {'State-action pairs':>20}")
for name, (n_states, n_pairs) in coverage.items():
    print(f"{name:<16} {n_states:>8} {n_pairs:>20}")
print(f"{'Total':<16} {abstraction_mapper.abstract_n_states:>8} {n_state_action_pairs:>20}")

Policy             States   State-action pairs
Randomized             92                  265
QValue                162                  463
ActiveLearning        150                  439
EntropyLearning       294                  882
Custom                126                  244
Total                 400                 1200


Unsurprisingly, achieves similar coverage to the 